In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModelForSequenceClassification, DistilBertModel, TextClassificationPipeline
from datasets import load_dataset
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm, trange

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import einops
from fancy_einsum import einsum
import tqdm.notebook as tqdm
import random
from pathlib import Path
import plotly.express as px
from torch.utils.data import DataLoader

from typing import List, Union, Optional
from functools import partial
import copy

import itertools
from transformers import AutoModelForCausalLM, AutoConfig, AutoTokenizer
import dataclasses
import datasets
from IPython.display import HTML

In [38]:
import torch
from transformers import RobertaForSequenceClassification, RobertaTokenizer

# Load pretrained model
model_name = "s-nlp/roberta_toxicity_classifier"  # Replace with your fine-tuned model path if needed
tokenizer = RobertaTokenizer.from_pretrained('s-nlp/roberta_toxicity_classifier')
model = RobertaForSequenceClassification.from_pretrained('s-nlp/roberta_toxicity_classifier')

Some weights of the model checkpoint at s-nlp/roberta_toxicity_classifier were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [26]:
# Run a forward pass
text = "This is a test input."
inputs = tokenizer(text, return_tensors="pt")
with torch.no_grad():
    outputs = model(**inputs)

In [27]:
outputs.logits

tensor([[ 4.9455, -5.1598]])

In [36]:
hook.remove()

In [40]:
model.roberta.encoder

RobertaEncoder(
  (layer): ModuleList(
    (0-11): 12 x RobertaLayer(
      (attention): RobertaAttention(
        (self): RobertaSdpaSelfAttention(
          (query): Linear(in_features=768, out_features=768, bias=True)
          (key): Linear(in_features=768, out_features=768, bias=True)
          (value): Linear(in_features=768, out_features=768, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (output): RobertaSelfOutput(
          (dense): Linear(in_features=768, out_features=768, bias=True)
          (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
      (intermediate): RobertaIntermediate(
        (dense): Linear(in_features=768, out_features=3072, bias=True)
        (intermediate_act_fn): GELUActivation()
      )
      (output): RobertaOutput(
        (dense): Linear(in_features=3072, out_features=768, bias=True)
        (LayerNorm): LayerNorm((768,), eps=1e-05,

In [39]:
# Select layer and head to ablate
target_layer = 4  # Example: 5th layer (indexing starts at 0)
target_head = 2   # Example: 3rd attention head

def hook_fn(module, input, output):
    """
    Hook function to modify attention outputs.
    """
    # print(len(output))
    # print(output[0][0].shape)
    # attn_output, attn_weights = output  # Output is a tuple (values, attention scores)
    
    # Zero out the contribution of the specific head
    output[:, target_head, :, :] = 0  # Shape: (batch, num_heads, seq_len, head_dim)

    return output

# Register hook on the target layer's attention module
hook = model.roberta.encoder.layer[target_layer].attention.self.register_forward_hook(hook_fn)

# Run a forward pass
text = "This is a test input."
inputs = tokenizer(text, return_tensors="pt")
with torch.no_grad():
    outputs = model(**inputs)

# Remove the hook after modification
hook.remove()
 

TypeError: 'tuple' object does not support item assignment

In [52]:
from datasets import load_dataset

In [53]:
dataset = load_dataset("imdb")

In [51]:
import torch
from transformers import RobertaModel, RobertaTokenizer

# Load model
model = RobertaModel.from_pretrained("s-nlp/roberta_toxicity_classifier")
tokenizer = RobertaTokenizer.from_pretrained("s-nlp/roberta_toxicity_classifier")

def zero_out_attention_head(module, input, output):
    """
    Hook function to zero out a specific attention head's contribution.
    - 'output[1]' contains attention scores before applying to values.
    """
    head_index = 2  # Change this to the head you want to zero out

    # Attention scores before applying to values (softmax output)
    attention_probs = output[1]  # Shape: (batch_size, num_heads, seq_len, seq_len)

    # Set specific head to zero
    attention_probs[:, head_index, :, :] = 0  

    return output  # Return modified output

# Register hook in self-attention module of a specific layer
layer_idx = 6  # Choose which layer to modify
hook = model.encoder.layer[layer_idx].attention.self.register_forward_hook(zero_out_attention_head)

# Test input
text = "Hello, how are you?"
inputs = tokenizer(text, return_tensors="pt")

# Run inference
with torch.no_grad():
    outputs = model(**inputs)

# Remove hook
hook.remove()


IndexError: tuple index out of range

In [50]:
outputs

BaseModelOutputWithPoolingAndCrossAttentions(last_hidden_state=tensor([[[-1.1054,  1.2266,  1.1682,  ...,  1.0640,  0.6632,  0.1873],
         [-1.4264,  1.3281,  1.2764,  ...,  1.3966,  0.7282,  0.0333],
         [-1.4482,  1.4591,  1.0689,  ...,  1.2197,  0.9050,  0.1216],
         ...,
         [-1.1112,  1.4137,  1.3164,  ...,  1.5241,  0.7260,  0.2660],
         [-1.2375,  1.4878,  1.2795,  ...,  1.3482,  0.6871,  0.1335],
         [-1.0624,  0.9808,  1.0491,  ...,  1.4277,  0.5301,  0.4266]]]), pooler_output=tensor([[nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
         nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
         nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
         nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,